In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
df = pd.read_csv('./data/premium.csv')
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [8]:
df = df.drop_duplicates()
df.info()

<class 'pandas.DataFrame'>
Index: 1337 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1337 non-null   int64  
 1   sex       1337 non-null   str    
 2   bmi       1332 non-null   float64
 3   children  1337 non-null   int64  
 4   smoker    1337 non-null   str    
 5   region    1337 non-null   str    
 6   charges   1337 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 83.6 KB


In [11]:
print(df.bmi.isnull())

0       False
1       False
2       False
3       False
4       False
        ...  
1333    False
1334    False
1335    False
1336    False
1337    False
Name: bmi, Length: 1337, dtype: bool


In [12]:
df['bmi'] = df['bmi'].fillna(df['bmi'].mean())

In [13]:
df.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [14]:
from sklearn.preprocessing import LabelEncoder

In [16]:
# 문자열 데이터의 수치화 > LabelEncoder
col_list = ['sex', 'smoker', 'region']
for col in col_list: 
  enc = LabelEncoder()
  df[col] = enc.fit_transform(df[col])
  
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,0,27.900,0,1,3,16884.92400
1,18,1,33.770,1,0,2,1725.55230
2,28,1,33.000,3,0,2,4449.46200
3,33,1,22.705,0,0,1,21984.47061
4,32,1,28.880,0,0,1,3866.85520


In [ ]:
# 스케일링 하기


In [17]:
X = df.iloc[:,:-1]
y = df.iloc[:, -1]

In [20]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2, random_state=42)

In [27]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train['bmi'] = scaler.fit_transform(X_train[['bmi']])
X_test['bmi'] = scaler.transform(X_test[['bmi']])
X_test['bmi']


900    -1.330419
1064   -0.818614
1256    0.970629
298     0.639656
237     1.303260
          ...   
534     1.649993
542     0.956527
760     0.671177
1284    0.956527
1285   -1.030968
Name: bmi, Length: 268, dtype: float64

In [31]:
# 선형회귀 모델
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred = lr_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
mae, mse, rmse, r2

(4170.6423560015255,
 35550130.517054714,
 np.float64(5962.393019338352),
 0.8065362865570331)

In [32]:
# 다항회귀

from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

In [42]:
degree = [2, 3, 4]

for deg in degree:
    model_poly = Pipeline([
        ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
        ('linear', LinearRegression())
    ])
    
    model_poly.fit(X_train, y_train)
    poly_pred = model_poly.predict(X_test)
    
    mae = mean_absolute_error(y_test, poly_pred)
    mse = mean_squared_error(y_test, poly_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, poly_pred)
    
    print(f'Degree {degree} | MAE: {mae:.4f} | MSE: {mse:.4f} | RMSE: {rmse:.4f} | R^2: {r2:.4f}')

Degree [2, 3, 4] | MAE: 2838.2869 | MSE: 20879449.3187 | RMSE: 4569.4036 | R^2: 0.8864
Degree [2, 3, 4] | MAE: 3021.4520 | MSE: 22521905.9297 | RMSE: 4745.7250 | R^2: 0.8774
Degree [2, 3, 4] | MAE: 3245.7158 | MSE: 26691158.4155 | RMSE: 5166.3487 | R^2: 0.8547


In [45]:
from sklearn.ensemble import RandomForestRegressor
model_rf = RandomForestRegressor(n_estimators=100, random_state= 0)
model_rf.fit(X_train, y_train)
y_pred_rf = model_rf.predict(X_test)
mae = mean_absolute_error(y_test, y_pred_rf)
mse = mean_squared_error(y_test, y_pred_rf)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred_rf)
mae, mse, rmse, r2

(2597.4139290194025,
 21900427.796358738,
 np.float64(4679.7892897393085),
 0.8808179315842309)

In [48]:
import pandas as pd
feature_names = ['age', 'sex', 'bmi', 'children', 'smoker', 'region']
importance_df = pd.DataFrame({
  "feature" : feature_names,
  'importance' : model_rf.feature_importances_
}).sort_values('importance', ascending= False)

print('\n--- 특성 중요도 ---')
print(importance_df)


--- 특성 중요도 ---
    feature  importance
4    smoker    0.600497
2       bmi    0.212468
0       age    0.138651
3  children    0.024268
5    region    0.016831
1       sex    0.007285


In [52]:
#XGBRegressor
from xgboost import XGBRegressor

xgb_reg = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=0
)

xgb_reg.fit(X_train, y_train)
poly_pred = xgb_reg.predict(X_test)

mae = mean_absolute_error(y_test, poly_pred)
mse = mean_squared_error(y_test, poly_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, poly_pred)
    
print(f'MAE: {mae:.4f} | MSE: {mse:.4f} | RMSE: {rmse:.4f} | R^2: {r2:.4f}')

MAE: 2631.0079 | MSE: 19747411.6369 | RMSE: 4443.8060 | R^2: 0.8925
